In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('dataset.csv', index_col=0)
df = df.drop_duplicates(subset='track_id')
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [15]:
median_pop = df['popularity'].median()
print("Median popularity:", median_pop)

df['is_popular'] = (df['popularity'] >= median_pop).astype(int)
df['is_popular'].value_counts()

Median popularity: 33.0


is_popular
1    45775
0    43966
Name: count, dtype: int64

In [16]:
features = ['danceability','energy','loudness','speechiness','acousticness',
            'instrumentalness','liveness','valence','tempo','duration_ms']

X = df[features]
y = df['is_popular']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
# Train a Decision Tree
clf_dt = DecisionTreeClassifier(max_depth=8, random_state=42)
clf_dt.fit(X_train, y_train)

train_score = clf_dt.score(X_train, y_train)
test_score = clf_dt.score(X_test, y_test)

print("Decision Tree Train Accuracy:", round(train_score, 4))
print("Decision Tree Test Accuracy:", round(test_score, 4))

Decision Tree Train Accuracy: 0.6389
Decision Tree Test Accuracy: 0.6231


In [18]:
# Train a Logistic Regression model
clf_log = LogisticRegression(max_iter=1000)
clf_log.fit(X_train, y_train)

train_score_log = clf_log.score(X_train, y_train)
test_score_log = clf_log.score(X_test, y_test)

print("Logistic Regression Train Accuracy:", round(train_score_log, 4))
print("Logistic Regression Test Accuracy:", round(test_score_log, 4))

Logistic Regression Train Accuracy: 0.5768
Logistic Regression Test Accuracy: 0.5756


In [19]:
# See which features the Decision Tree found most useful
importances = pd.Series(clf_dt.feature_importances_, index=features)
importances.sort_values(ascending=False)

acousticness        0.203697
instrumentalness    0.159786
speechiness         0.125017
valence             0.111821
duration_ms         0.094024
energy              0.087577
danceability        0.078738
loudness            0.050836
liveness            0.044899
tempo               0.043605
dtype: float64

In [20]:
df2 = pd.read_csv('Spotify_Youtube.csv', index_col=0)
df2 = df2.dropna(subset=['Views','Danceability','Energy','Loudness','Speechiness',
                          'Acousticness','Instrumentalness','Liveness','Valence','Tempo'])

median_views = df2['Views'].median()
df2['is_popular'] = (df2['Views'] >= median_views).astype(int)

features2 = ['Danceability','Energy','Loudness','Speechiness','Acousticness',
             'Instrumentalness','Liveness','Valence','Tempo']

X2 = df2[features2]
y2 = df2['is_popular']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

clf_dt2 = DecisionTreeClassifier(max_depth=8, random_state=42)
clf_dt2.fit(X2_train, y2_train)
print("Dataset 2 Decision Tree Test Accuracy:", round(clf_dt2.score(X2_test, y2_test), 4))

clf_log2 = LogisticRegression(max_iter=1000)
clf_log2.fit(X2_train, y2_train)
print("Dataset 2 Logistic Regression Test Accuracy:", round(clf_log2.score(X2_test, y2_test), 4))

Dataset 2 Decision Tree Test Accuracy: 0.5857
Dataset 2 Logistic Regression Test Accuracy: 0.6005


In [21]:
# Get predictions so we can look at more detailed metrics
dt_pred = clf_dt.predict(X_test)
log_pred = clf_log.predict(X_test)

print("Decision Tree - Precision:", round(precision_score(y_test, dt_pred), 4))
print("Decision Tree - Recall:", round(recall_score(y_test, dt_pred), 4))
print("Decision Tree - F1:", round(f1_score(y_test, dt_pred), 4))
print()
print("Logistic Regression - Precision:", round(precision_score(y_test, log_pred), 4))
print("Logistic Regression - Recall:", round(recall_score(y_test, log_pred), 4))
print("Logistic Regression - F1:", round(f1_score(y_test, log_pred), 4))

Decision Tree - Precision: 0.6054
Decision Tree - Recall: 0.7511
Decision Tree - F1: 0.6705

Logistic Regression - Precision: 0.5721
Logistic Regression - Recall: 0.6694
Logistic Regression - F1: 0.6169


In [12]:
dt2_pred = clf_dt2.predict(X2_test)
log2_pred = clf_log2.predict(X2_test)

print("Dataset 2 Decision Tree - Precision:", round(precision_score(y2_test, dt2_pred), 4))
print("Dataset 2 Decision Tree - Recall:", round(recall_score(y2_test, dt2_pred), 4))
print("Dataset 2 Decision Tree - F1:", round(f1_score(y2_test, dt2_pred), 4))
print()
print("Dataset 2 Logistic Regression - Precision:", round(precision_score(y2_test, log2_pred), 4))
print("Dataset 2 Logistic Regression - Recall:", round(recall_score(y2_test, log2_pred), 4))
print("Dataset 2 Logistic Regression - F1:", round(f1_score(y2_test, log2_pred), 4))

Dataset 2 Decision Tree - Precision: 0.5708
Dataset 2 Decision Tree - Recall: 0.664
Dataset 2 Decision Tree - F1: 0.6139

Dataset 2 Logistic Regression - Precision: 0.5776
Dataset 2 Logistic Regression - Recall: 0.7242
Dataset 2 Logistic Regression - F1: 0.6427


In [13]:
importances2 = pd.Series(clf_dt2.feature_importances_, index=features2)
importances2.sort_values(ascending=False)

Loudness            0.452988
Instrumentalness    0.107095
Energy              0.077427
Danceability        0.075940
Speechiness         0.071473
Tempo               0.065494
Liveness            0.064053
Valence             0.045545
Acousticness        0.039985
dtype: float64